# RAG Pipeline Evaluation
**Author:** Juan Esteban Agudelo Ortiz  
**Email:** juan.es.agor@gmail.com

---

This notebook evaluates the RAG pipeline built in the previous notebooks
using three standard metrics implemented from first principles. Evaluation
uses a judge model (Qwen2.5-7B) separate from the generator model
(Qwen2.5-3B) to avoid self-evaluation bias.

A RAG pipeline has two components that can fail independently: the retriever
and the generator. The three metrics isolate each component, allowing
targeted diagnosis of where the pipeline underperforms.

### Limitations
1. Evaluation uses a synthetic dataset generated by the judge model; questions
   may not reflect the actual queries a real user would ask.
2. Metrics are estimated by the judge model, which introduces evaluation bias
   despite using a separate model from the generator.
3. Evaluation is slow on CPU; expect 2-5 minutes per sample.
4. Results are specific to the evaluation dataset; performance may vary on
   out-of-distribution queries.

## 0. Install dependencies

In [26]:
# Run only once
# !pip install llama-index-core llama-index-embeddings-huggingface chromadb llama-index-vector-stores-chroma llama-cpp-python

## 1. Imports and configuration

In [27]:
import json
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum

import chromadb
from llama_index.core import VectorStoreIndex, StorageContext, Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_cpp import Llama

# --- Directory setup ---
BASE_DIR    = Path("..")
UPLOADS_DIR = BASE_DIR / "data" / "uploads"
OUT_DIR     = BASE_DIR / "outputs"
INDEX_DIR   = BASE_DIR / "data" / "index"
MODELS_DIR  = BASE_DIR / "data" / "models"

UPLOADS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Constants ---
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
GENERATOR_MODEL      = "qwen2.5-3b-instruct-q4_k_m.gguf"
GENERATOR_PATH       = MODELS_DIR / GENERATOR_MODEL
JUDGE_MODEL_PARTS = [
    "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf",
]
JUDGE_PATH = MODELS_DIR / JUDGE_MODEL_PARTS[0]

# --- Reproduce IngestedDocument from notebook 1 ---
class InputType(Enum):
    PDF           = "pdf"
    HANDWRITTEN   = "handwritten_image"
    REFERENCE_IMG = "reference_image"
    PLAIN_TEXT    = "plain_text"

@dataclass
class IngestedDocument:
    input_type       : InputType
    text             : str
    reference_images : list = field(default_factory=list)
    source_path      : Optional[Path] = None

print(f"Uploads dir : {UPLOADS_DIR.resolve()}")
print(f"Models dir  : {MODELS_DIR.resolve()}")
print(f"Index dir   : {INDEX_DIR.resolve()}")
print("Configuration ready ✓")

Uploads dir : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/uploads
Models dir  : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/models
Index dir   : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/index
Configuration ready ✓


## 2. Evaluation dataset construction

The evaluation dataset consists of question-answer pairs generated
synthetically from the document chunks. The judge model generates a
question and a ground truth answer from each chunk. The RAG pipeline
then produces an answer and retrieves context for each question.

Each sample contains four fields:

| Field | Description |
|---|---|
| `question` | Question generated from a chunk by the judge model |
| `ground_truth` | Expected answer extracted from the chunk by the judge model |
| `answer` | Answer produced by the RAG pipeline for that question |
| `contexts` | Chunks retrieved by the retriever for that question |

The judge model generates `question` and `ground_truth` independently
of the RAG pipeline. This independence is critical: if the same pipeline
generated both `answer` and `ground_truth`, the evaluation would measure
self-consistency rather than correctness.

In [28]:
DATASET_GENERATION_PROMPT = """Given the following text chunk from a textbook, generate a question and its answer.

The question must:
- Be answerable using only the information in the chunk
- Be specific enough that there is one clear correct answer
- Be phrased as a student would ask it

Respond ONLY with a JSON object with no preamble or markdown backticks:
{{
    "question": "the question",
    "ground_truth": "the answer based strictly on the chunk"
}}

Chunk:
{chunk}"""


def generate_eval_sample(
    chunk: str,
    judge_llm: Llama,
    max_tokens: int = 256,
) -> dict | None:
    """
    Generate a question-answer pair from a text chunk using the judge model.

    Parameters
    ----------
    chunk : str
        Text chunk to generate a question from.
    judge_llm : Llama
        Judge language model (Qwen2.5-7B).
    max_tokens : int
        Maximum tokens to generate. Default 256.

    Returns
    -------
    dict or None
        Dict with 'question' and 'ground_truth' keys, or None if generation fails.
    """
    prompt = DATASET_GENERATION_PROMPT.format(chunk=chunk)

    response = judge_llm.create_chat_completion(
        messages    = [{"role": "user", "content": prompt}],
        max_tokens  = max_tokens,
        temperature = 0.2,
    )

    raw = response["choices"][0]["message"]["content"].strip()

    try:
        parsed = json.loads(raw)
        if "question" in parsed and "ground_truth" in parsed:
            return parsed
        return None
    except json.JSONDecodeError:
        return None


def build_eval_dataset(
    chunks: list[dict],
    judge_llm: Llama,
    n_samples: int = 10,
) -> list[dict]:
    """
    Build a synthetic evaluation dataset from document chunks.

    Parameters
    ----------
    chunks : list[dict]
        Chunks from the chunking pipeline.
    judge_llm : Llama
        Judge language model for question generation.
    n_samples : int
        Number of samples to generate. Default 10.

    Returns
    -------
    list[dict]
        List of samples with 'question' and 'ground_truth' fields.
    """
    import random
    selected = random.sample(chunks, min(n_samples, len(chunks)))

    dataset = []
    for i, chunk in enumerate(selected):
        print(f"Generating sample {i+1}/{len(selected)}...")
        sample = generate_eval_sample(chunk["text"], judge_llm)
        if sample is not None:
            sample["source_chunk"] = chunk["text"]
            dataset.append(sample)

    print(f"\nDataset built: {len(dataset)} samples generated")
    return dataset


print("build_eval_dataset defined ✓")

build_eval_dataset defined ✓


## 3. Evaluation metrics

Three metrics evaluate different aspects of the RAG pipeline:

**Faithfulness** measures whether the generated answer is grounded in
the retrieved context. A faithful answer contains only claims that can
be inferred from the context, not from the model's parametric knowledge.

$$\text{Faithfulness} = \frac{\text{claims in answer supported by context}}{\text{total claims in answer}}$$

**Answer Relevancy** measures whether the generated answer addresses
the question asked. A relevant answer is direct and complete with
respect to the question.

$$\text{Answer Relevancy} = \frac{\text{aspects of the question addressed}}{\text{total aspects of the question}}$$

**Context Precision** measures whether the retrieved chunks contain
the information needed to answer the question. A precise retrieval
returns relevant chunks without noise.

$$\text{Context Precision} = \frac{\text{retrieved chunks relevant to the question}}{\text{total chunks retrieved}}$$

All three metrics are estimated by the judge model on a scale $[0, 1]$.
Higher is better in all cases.

One important thing to note here is that the model has `temperature=0.0`, and that is because we want the responses to the metrics always to be the same for the same prompt, i.e., reproducibility. Also, here this variance will not risk falling into a token-infinite-loop trying to generate the same probable next token after the current one since the answer is short and structured, mitigating this risks.

In [29]:
FAITHFULNESS_PROMPT = """You are evaluating whether an answer is faithful to the provided context.

A faithful answer contains only claims that can be directly inferred from the context.
An unfaithful answer introduces information not present in the context.

Context:
{context}

Answer:
{answer}

Evaluate each claim in the answer and determine if it is supported by the context.
Respond ONLY with a JSON object:
{{
    "supported_claims": <number of claims supported by context>,
    "total_claims": <total number of claims in the answer>,
    "reasoning": "brief explanation"
}}"""


ANSWER_RELEVANCY_PROMPT = """You are evaluating whether an answer is relevant to the question asked.

A relevant answer directly addresses the question without unnecessary information.

Question:
{question}

Answer:
{answer}

Evaluate how many aspects of the question are addressed by the answer.
Respond ONLY with a JSON object:
{{
    "addressed_aspects": <number of question aspects addressed>,
    "total_aspects": <total number of aspects in the question>,
    "reasoning": "brief explanation"
}}"""


CONTEXT_PRECISION_PROMPT = """You are evaluating whether retrieved context chunks are relevant to a question.

Question:
{question}

Retrieved chunks:
{contexts}

Evaluate how many of the retrieved chunks contain information relevant to answering the question.
Respond ONLY with a JSON object:
{{
    "relevant_chunks": <number of relevant chunks>,
    "total_chunks": <total number of chunks>,
    "reasoning": "brief explanation"
}}"""


def evaluate_metric(
    prompt: str,
    judge_llm: Llama,
    max_tokens: int = 256,
) -> dict | None:
    """
    Run a single metric evaluation using the judge model.

    Parameters
    ----------
    prompt : str
        Formatted evaluation prompt.
    judge_llm : Llama
        Judge language model.
    max_tokens : int
        Maximum tokens to generate.

    Returns
    -------
    dict or None
        Parsed evaluation result or None if parsing fails.
    """
    response = judge_llm.create_chat_completion(
        messages    = [{"role": "user", "content": prompt}],
        max_tokens  = max_tokens,
        temperature = 0.0,
    )

    raw = response["choices"][0]["message"]["content"].strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None


def compute_metrics(
    question: str,
    answer: str,
    contexts: list[str],
    judge_llm: Llama,
) -> dict:
    """
    Compute faithfulness, answer relevancy, and context precision for one sample.

    Parameters
    ----------
    question : str
        The evaluation question.
    answer : str
        The answer produced by the RAG pipeline.
    contexts : list[str]
        The chunks retrieved by the retriever.
    judge_llm : Llama
        Judge language model.

    Returns
    -------
    dict
        Metric scores for this sample.
    """
    context_str = "\n\n".join(f"[Chunk {i+1}]\n{c}" for i, c in enumerate(contexts))

    results = {}

    # Faithfulness
    faith_result = evaluate_metric(
        FAITHFULNESS_PROMPT.format(context=context_str, answer=answer),
        judge_llm,
    )
    if faith_result and faith_result["total_claims"] > 0:
        results["faithfulness"] = faith_result["supported_claims"] / faith_result["total_claims"]
    else:
        results["faithfulness"] = None

    # Answer relevancy
    rel_result = evaluate_metric(
        ANSWER_RELEVANCY_PROMPT.format(question=question, answer=answer),
        judge_llm,
    )
    if rel_result and rel_result["total_aspects"] > 0:
        results["answer_relevancy"] = rel_result["addressed_aspects"] / rel_result["total_aspects"]
    else:
        results["answer_relevancy"] = None

    # Context precision
    prec_result = evaluate_metric(
        CONTEXT_PRECISION_PROMPT.format(question=question, contexts=context_str),
        judge_llm,
    )
    if prec_result and prec_result["total_chunks"] > 0:
        results["context_precision"] = prec_result["relevant_chunks"] / prec_result["total_chunks"]
    else:
        results["context_precision"] = None

    return results


print("Evaluation metrics defined ✓")

Evaluation metrics defined ✓


## 4. Running the evaluation

The evaluation loop runs the full pipeline for each sample in the
evaluation dataset:

1. Retrieve context chunks for the question using the RAG retriever
2. Generate an answer using the generator model (Qwen2.5-3B)
3. Compute the three metrics using the judge model (Qwen2.5-7B)

Each sample produces a score between 0 and 1 for each metric.
The final scores are averaged across all samples.

Note: evaluation is slow on CPU. With 10 samples and two model calls
per sample, expect 20-40 minutes of total runtime.

It is important to note that we are using a system prompt that is simpler than the one from previous notebooks because we are interested specifically on the answer given by the model, while in the previous prompts it included the output schema which is not what we want to test here. Additionally, we will be charging the models on sequence to avoid having both of them in the same computer in case the hardware is not enough for having them both uploaded at the same time.

In [30]:
def generate_answers(
    dataset: list[dict],
    index: VectorStoreIndex,
    generator_llm: Llama,
    embed_model: HuggingFaceEmbedding,
    k: int = 3,
    max_tokens: int = 512,
) -> list[dict]:
    """
    Generate answers for all evaluation samples using the generator model.
    Results are returned as a list to be saved to disk before loading the judge.
    """
    answers = []
    for i, sample in enumerate(dataset):
        print(f"Generating answer {i+1}/{len(dataset)}...")
        retriever = index.as_retriever(similarity_top_k=k, embed_model=embed_model)
        nodes     = retriever.retrieve(sample["question"])
        contexts  = [node.text for node in nodes]

        messages = [
            {"role": "system", "content": "You are a helpful study assistant. Answer the question based strictly on the provided context."},
            {"role": "user",   "content": f"Context:\n{chr(10).join(contexts)}\n\nQuestion: {sample['question']}"},
        ]
        response = generator_llm.create_chat_completion(
            messages    = messages,
            max_tokens  = max_tokens,
            temperature = 0.1,
        )
        answers.append({
            "question"     : sample["question"],
            "ground_truth" : sample["ground_truth"],
            "answer"       : response["choices"][0]["message"]["content"].strip(),
            "contexts"     : contexts,
        })
    return answers


def evaluate_answers(
    answers: list[dict],
    judge_llm: Llama,
) -> list[dict]:
    """
    Evaluate pre-generated answers using the judge model.
    Accepts answers loaded from disk after generator has been unloaded.
    """
    results = []
    for i, sample in enumerate(answers):
        print(f"\nEvaluating sample {i+1}/{len(answers)}")
        print(f"Question: {sample['question'][:80]}...")
        metrics = compute_metrics(
            question  = sample["question"],
            answer    = sample["answer"],
            contexts  = sample["contexts"],
            judge_llm = judge_llm,
        )
        results.append({**sample, **metrics})
        print(f"Faithfulness     : {metrics['faithfulness']}")
        print(f"Answer relevancy : {metrics['answer_relevancy']}")
        print(f"Context precision: {metrics['context_precision']}")
    return results


print("generate_answers and evaluate_answers defined ✓")

generate_answers and evaluate_answers defined ✓


## 5. Results analysis

The evaluation results are aggregated across all samples to produce
mean scores for each metric. Scores are interpreted as follows:

| Score | Interpretation |
|---|---|
| $> 0.8$ | Good — the component performs well |
| $0.5 - 0.8$ | Acceptable — room for improvement |
| $< 0.5$ | Poor — the component needs attention |

A low **context precision** score points to the retriever as the
bottleneck — the chunks recovered are not relevant enough.

A low **faithfulness** score points to the generator as the
bottleneck — the model is introducing information not in the context.

A low **answer relevancy** score indicates the generator is not
addressing the question directly.

In [31]:
def analyze_results(results: list[dict]) -> dict:
    """
    Aggregate evaluation results and compute mean scores per metric.

    Parameters
    ----------
    results : list[dict]
        Output of run_evaluation.

    Returns
    -------
    dict
        Mean scores and interpretation for each metric.
    """
    import numpy as np

    metrics = ["faithfulness", "answer_relevancy", "context_precision"]
    summary = {}

    print("=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)

    for metric in metrics:
        scores = [r[metric] for r in results if r[metric] is not None]

        if not scores:
            print(f"{metric:25s} : N/A (all samples failed)")
            summary[metric] = None
            continue

        mean  = float(np.mean(scores))
        std   = float(np.std(scores))

        if mean > 0.8:
            interpretation = "Good"
        elif mean >= 0.5:
            interpretation = "Acceptable"
        else:
            interpretation = "Poor — needs attention"

        print(f"{metric:25s} : {mean:.3f} ± {std:.3f} ({interpretation})")
        summary[metric] = {"mean": mean, "std": std, "interpretation": interpretation}

    print("=" * 50)
    print(f"\nSamples evaluated : {len(results)}")
    print(f"Samples failed    : {sum(1 for r in results if any(r[m] is None for m in metrics))}")

    return summary


print("analyze_results defined ✓")

analyze_results defined ✓


## 6. Demo — evaluating the RAG pipeline on OpenStax chapter 21

This demo runs the full evaluation pipeline on the carboxylic acid
derivatives chapter from OpenStax Organic Chemistry. The judge model
generates 10 evaluation samples from the document chunks, the RAG
pipeline answers each question, and the three metrics are computed
and aggregated.

Note: this demo requires both models to be loaded simultaneously.
Total RAM usage is approximately 7GB (2.5GB for Qwen2.5-3B + 4.5GB
for Qwen2.5-7B). Close other applications before running.

### Getting the PDF
Download the full book from:
```text
https://openstax.org/details/books/organic-chemistry
```
Then extract chapter 21 (pages 741 to 792) using PyMuPDF:

```python
import fitz
doc = fitz.open("openstax_organic_chemistry.pdf")
sub = fitz.open()
sub.insert_pdf(doc, from_page=740, to_page=791)
sub.save("data/uploads/openstax_ch21_carboxylic_acid_derivatives.pdf")
```

Note: PyMuPDF uses zero-based page indexing, so page 741 corresponds to index 740.

In [32]:
import fitz
import re
import gc
from llama_index.core.node_parser import SentenceSplitter

# --- Reproduce pipeline from previous notebooks ---
def extract_text_from_pdf(pdf_path: Path, min_block_chars: int = 20) -> str:
    """Extract ordered text from PDF. Reproduced from ingestion notebook."""
    doc = fitz.open(pdf_path)
    all_text = []
    for page_num, page in enumerate(doc):
        blocks = page.get_text("blocks")
        blocks_sorted = sorted(blocks, key=lambda b: (b[1], b[0]))
        page_text = []
        for block in blocks_sorted:
            text = block[4].strip()
            if len(text) < min_block_chars:
                continue
            text = re.sub(r"\s+", " ", text)
            page_text.append(text)
        if page_text:
            all_text.append(f"--- Page {page_num + 1} ---\n" + "\n\n".join(page_text))
    doc.close()
    return "\n\n".join(all_text)


def fixed_size_chunking(doc: IngestedDocument, chunk_size: int = 512, chunk_overlap: int = 64) -> list[dict]:
    """Fixed-size chunking. Reproduced from chunking notebook."""
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    llama_doc = Document(text=doc.text)
    nodes = splitter.get_nodes_from_documents([llama_doc])
    return [{"text": node.text, "index": i, "strategy": "fixed_size"} for i, node in enumerate(nodes)]


METADATA_FILE = "index_metadata.json"

def create_vector_store(collection_name: str, persist_dir: Path):
    chroma_client = chromadb.PersistentClient(path=str(persist_dir))
    collection    = chroma_client.get_or_create_collection(collection_name)
    vector_store  = ChromaVectorStore(chroma_collection=collection)
    return collection, vector_store

def load_and_verify_index(persist_dir, collection_name, current_model_name, embed_model, chunks):
    metadata_path = persist_dir / METADATA_FILE
    if metadata_path.exists():
        stored = json.loads(metadata_path.read_text(encoding="utf-8"))
        if stored.get("embedding_model") != current_model_name:
            import shutil
            shutil.rmtree(persist_dir)
            persist_dir.mkdir(parents=True, exist_ok=True)
        else:
            _, vector_store = create_vector_store(collection_name, persist_dir)
            return VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)
    _, vector_store = create_vector_store(collection_name, persist_dir)
    index = VectorStoreIndex.from_documents(
        [Document(text=c["text"], metadata={"chunk_index": c["index"], "strategy": c["strategy"]}) for c in chunks],
        storage_context = StorageContext.from_defaults(vector_store=vector_store),
        embed_model     = embed_model,
        show_progress   = True,
    )
    (persist_dir / METADATA_FILE).write_text(json.dumps({"embedding_model": current_model_name}), encoding="utf-8")
    return index

def load_language_model(model_path: Path, n_ctx: int = 2048) -> Llama:
    """
    Load a GGUF model using llama-cpp-python.
    Reproduced from the RAG pipeline notebook.
    """
    import os
    n_threads = os.cpu_count()
    print(f"Loading model from : {model_path}")
    print(f"Context window     : {n_ctx} tokens")
    print(f"CPU threads        : {n_threads}")
    llm = Llama(
        model_path = str(model_path),
        n_ctx      = n_ctx,
        n_threads  = n_threads,
        verbose    = False,
    )
    print("Language model ready ✓")
    return llm


from huggingface_hub import hf_hub_download

# Download generator model if not present
if not GENERATOR_PATH.exists():
    print(f"Downloading {GENERATOR_MODEL} (~2GB)...")
    hf_hub_download(
        repo_id   = "Qwen/Qwen2.5-3B-Instruct-GGUF",
        filename  = GENERATOR_MODEL,
        local_dir = str(MODELS_DIR),
    )
    print("Generator download complete ✓")
else:
    print(f"Generator model present ✓")


JUDGE_MODEL_PARTS = [
    "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf",
]
JUDGE_PATH = MODELS_DIR / JUDGE_MODEL_PARTS[0]

if not JUDGE_PATH.exists():
    print("Downloading Qwen2.5-7B Q4_K_M in 2 parts (~4.7GB total)...")
    for part in JUDGE_MODEL_PARTS:
        print(f"Downloading {part}...")
        hf_hub_download(
            repo_id   = "Qwen/Qwen2.5-7B-Instruct-GGUF",
            filename  = part,
            local_dir = str(MODELS_DIR),
        )
    print("Download complete ✓")
else:
    print(f"Judge model already present: {JUDGE_PATH}")

Generator model present ✓
Judge model already present: ../data/models/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf


In [34]:
# --- Step 1: setup index ---
pdf_path = UPLOADS_DIR / "openstax_ch21_carboxylic_acid_derivatives.pdf"

embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL_NAME, device="cpu")

doc = IngestedDocument(
    input_type  = InputType.PDF,
    text        = extract_text_from_pdf(pdf_path),
    source_path = pdf_path,
)

chunks = fixed_size_chunking(doc)
index  = load_and_verify_index(
    persist_dir        = INDEX_DIR / "ch21",
    collection_name    = "openstax_ch21",
    current_model_name = EMBEDDING_MODEL_NAME,
    embed_model        = embed_model,
    chunks             = chunks,
)

# --- Step 2: load generator, build dataset and generate answers ---
print("\nLoading generator model (Qwen2.5-3B)...")
generator_llm = load_language_model(GENERATOR_PATH)

print("\nGenerating evaluation dataset...")
dataset = build_eval_dataset(chunks, generator_llm, n_samples=5)

print("\nGenerating answers...")
answers = generate_answers(
    dataset       = dataset,
    index         = index,
    generator_llm = generator_llm,
    embed_model   = embed_model,
)

# Save answers to disk before unloading generator
answers_path = OUT_DIR / "eval_answers.json"
answers_path.write_text(json.dumps(answers, indent=2), encoding="utf-8")
print(f"Answers saved to {answers_path}")

# Unload generator and free memory
del generator_llm
gc.collect()
print("Generator unloaded ✓")

# --- Step 3: load judge and evaluate ---
print("\nLoading judge model (Qwen2.5-7B)...")
judge_llm = load_language_model(JUDGE_PATH, n_ctx=4096)

print("\nEvaluating answers...")
answers_loaded = json.loads(answers_path.read_text(encoding="utf-8"))
results        = evaluate_answers(answers_loaded, judge_llm)

# --- Step 4: analyze and save results ---
summary      = analyze_results(results)
results_path = OUT_DIR / "eval_results.json"
results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"\nFull results saved to {results_path}")

# Unload judge and free memory
del judge_llm
gc.collect()
print("Judge unloaded ✓")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading generator model (Qwen2.5-3B)...
Loading model from : ../data/models/qwen2.5-3b-instruct-q4_k_m.gguf
Context window     : 2048 tokens
CPU threads        : 8


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Language model ready ✓

Generating evaluation dataset...
Generating sample 1/5...
Generating sample 2/5...
Generating sample 3/5...
Generating sample 4/5...
Generating sample 5/5...

Dataset built: 5 samples generated

Generating answers...
Generating answer 1/5...
Generating answer 2/5...
Generating answer 3/5...
Generating answer 4/5...
Generating answer 5/5...
Answers saved to ../outputs/eval_answers.json
Generator unloaded ✓

Loading judge model (Qwen2.5-7B)...
Loading model from : ../data/models/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
Context window     : 4096 tokens
CPU threads        : 8


llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Language model ready ✓

Evaluating answers...

Evaluating sample 1/5
Question: What is the mechanism of activation of a carboxylic acid in biological chemistry...
Faithfulness     : 0.6666666666666666
Answer relevancy : 1.0
Context precision: 0.3333333333333333

Evaluating sample 2/5
Question: Why is the saponification of an ester irreversible?...
Faithfulness     : 0.6
Answer relevancy : 1.0
Context precision: 0.3333333333333333

Evaluating sample 3/5
Question: What is the carbon number for the 2° carbon in the 13C NMR spectrum of C7H12O2?...
Faithfulness     : 1.0
Answer relevancy : 0.5
Context precision: 0.3333333333333333

Evaluating sample 4/5
Question: What is wrong with the synthetic scheme for p-Aminobenzoic acid (PABA) starting ...
Faithfulness     : None
Answer relevancy : 0.5
Context precision: 0.3333333333333333

Evaluating sample 5/5
Question: What is the difference in behavior between aldehydes/ketones and carboxylic acid...
Faithfulness     : 0.6
Answer relevancy : 0.5
C